# Module 5, Demo lab: predicting frozen yogurt sales

Not graded, follow along. Prerequisite: [Module 5 Read 1](../lectures/module-5-read-01).

The question. You run Bluerock Yogurt, and you want to predict how many frozen yogurts you will sell in a day from that day's high temperature. Tomorrow's forecast high is 57 degrees Fahrenheit, and you want two things: a single best guess for tomorrow's sales, and a 95 percent interval that honestly shows how uncertain that guess is.

This notebook walks through the whole workflow on realistic data: get the data into R, explore it, build a simple linear regression, assess it, then use it to predict. These are the same moves you will make on your own data later, so go slowly and read every step.

## Code anatomy legend

In the demo notebooks we color, code how to read R:

| Color | Meaning in code |
|-------|-----------------|
| <span style="color:#2563eb">Blue</span> | R functions and syntax (mean, <-, (), ~) |
| <span style="color:#059669">Green</span> | Names you created (variables, data frames) |
| <span style="color:#dc2626">Red</span> | Values to change for your own question or data |
| <span style="color:#7c3aed">Purple</span> | Important output to read carefully |

Colab may add its own syntax colors in the editor; our anatomy colors are a reading guide in the tables below.

## Step 0, The data and a spreadsheet

Over the past two weeks you wrote down each day's high temperature and how many frozen yogurts you sold:

| date | high_temp_f | fro_yos_sold |
|------|-------------|--------------|
| 2026-06-01 | 52 | 56 |
| 2026-06-02 | 55 | 71 |
| 2026-06-03 | 49 | 49 |
| 2026-06-04 | 61 | 78 |
| 2026-06-05 | 64 | 92 |
| 2026-06-06 | 58 | 70 |
| 2026-06-07 | 53 | 66 |
| 2026-06-08 | 67 | 90 |
| 2026-06-09 | 71 | 99 |
| 2026-06-10 | 60 | 82 |
| 2026-06-11 | 56 | 64 |
| 2026-06-12 | 63 | 89 |
| 2026-06-13 | 50 | 58 |
| 2026-06-14 | 66 | 95 |

Put this in a spreadsheet (Google Sheets or Excel) with the variable names in the very first row. Use three columns in this order: column 1 is date, column 2 is high_temp_f (the high temperature in Fahrenheit), and column 3 is fro_yos_sold (the count sold that day). Use one row per day. Clean column names with no spaces make the R code easier.

## Step 1, Connect your spreadsheet to Colab

You have two easy ways to get a Google Sheet into R in Colab.

Option A, read straight from the sheet (recommended). This needs no login as long as the sheet is shared for viewing.

1. In Google Sheets, click Share, then set General access to Anyone with the link, Viewer.
2. Copy the link. It looks like `https://docs.google.com/spreadsheets/d/SHEET_ID/edit#gid=0`
3. Turn it into a CSV export link by replacing the end: `https://docs.google.com/spreadsheets/d/SHEET_ID/export?format=csv&gid=0`
4. Give that CSV link to `read_csv()`. This is the same idea as reading a raw CSV from any web URL.

Option B, download and upload (backup). In Sheets choose File, Download, Comma-separated values (.csv). Then in Colab open the Files pane (the folder icon on the left), click Upload, choose your file, and read it by its name, for example `read_csv("bluerock_yogurt.csv")`. Use this if a sheet will not share or you are working offline.

There is also an R package, googlesheets4, but it adds a sign-in step that is fiddly in Colab, so we use the simpler CSV link here.

### Code anatomy, loading and checking

| Part | Role |
|------|------|
| <span style="color:#2563eb">library()</span> | Load the tidyverse tools (read_csv, ggplot, dplyr) |
| <span style="color:#2563eb">read_csv()</span> | Read a CSV (from a URL or an uploaded file) into a data frame |
| <span style="color:#dc2626">sheet_csv_url</span> | Replace with your own sheet's CSV export link |
| <span style="color:#2563eb">glimpse()</span>, <span style="color:#2563eb">head()</span>, <span style="color:#2563eb">names()</span>, <span style="color:#2563eb">nrow()</span> | Inspect structure, first rows, column names, row count |

In [ ]:
if (!requireNamespace("tidyverse", quietly = TRUE)) {
  install.packages("tidyverse", repos = "https://cloud.r-project.org")
}
library(tidyverse)

In [ ]:
# Option A (recommended): read straight from the shared Google Sheet.
# This is the sheet's CSV export link (Share -> Anyone with the link -> Viewer,
# then change the end of the edit link to export?format=csv).
sheet_csv_url <- "https://docs.google.com/spreadsheets/d/1bn8QKxfBWb7evqGVYMowvSIPWvVgIElmDIm_9dyQjew/export?format=csv"

froyo <- read_csv(sheet_csv_url, show_col_types = FALSE)

# Option B (backup): after downloading the sheet and uploading the .csv with the
# Colab Files pane, you would instead use:
# froyo <- read_csv("bluerock_yogurt.csv", show_col_types = FALSE)

In [ ]:
glimpse(froyo)

head(froyo)

names(froyo)

nrow(froyo)

Read the output. <span style="color:#7c3aed">glimpse()</span> shows one line per column with its type: date should be a date, and high_temp_f and fro_yos_sold should be numbers (dbl). If a number came in as text (chr), a stray symbol or letter in the spreadsheet is usually the cause. <span style="color:#7c3aed">nrow()</span> should be 14, one row per day.

## Step 2, Explore the data

Before modeling, look at center, spread, and the relationship. The smallest and largest high_temp_f matter because they set the range where prediction is trustworthy: predicting inside that range is interpolation, predicting far outside it is extrapolation.

### Code anatomy, summaries and correlation

| Part | Role |
|------|------|
| <span style="color:#2563eb">summary()</span> | Quick summary of each column |
| <span style="color:#2563eb">summarise()</span>, <span style="color:#2563eb">mean()</span>, <span style="color:#2563eb">sd()</span>, <span style="color:#2563eb">min()</span>, <span style="color:#2563eb">max()</span> | Center, spread, and the range you report |
| <span style="color:#2563eb">cor()</span> | Pearson correlation r, strength and direction of a straight-line association |

In [ ]:
summary(froyo)

froyo |>
  summarise(
    mean_temp = mean(high_temp_f),
    sd_temp   = sd(high_temp_f),
    min_temp  = min(high_temp_f),
    max_temp  = max(high_temp_f),
    mean_sold = mean(fro_yos_sold),
    sd_sold   = sd(fro_yos_sold)
  )

cor(froyo$high_temp_f, froyo$fro_yos_sold)

Read the output. The high temperatures run from about 49 to 71 degrees (mean near 59), so a forecast of 57 sits comfortably inside the observed range, which is good news for predicting tomorrow. Sales average about 76 per day. <span style="color:#7c3aed">cor()</span> is about 0.97: a strong, positive, linear association, hotter days go with more frozen yogurt sold. A correlation this close to 1 says a straight line should describe the data well.

In [ ]:
ggplot(froyo, aes(x = high_temp_f, y = fro_yos_sold)) +
  geom_point(color = "steelblue4", size = 2) +
  geom_smooth(method = "lm", se = FALSE, color = "darkred") +
  labs(
    title = "Bluerock Yogurt: daily sales vs high temperature",
    x = "High temperature (F)",
    y = "Frozen yogurts sold"
  ) +
  theme_minimal()

Read the plot. Form: the points follow a straight band, not a curve, so a line is appropriate. Direction: positive, the band rises to the right. Strength: strong, the points sit close to the red least-squares line. This matches the correlation near 0.97 and tells us simple linear regression is a reasonable model.

## Step 3, Build the model

Fit a simple linear regression. Think carefully about roles: we want to predict sales from temperature, so the response y is fro_yos_sold and the predictor x is high_temp_f. In R the formula is written y ~ x, that is response ~ predictor.

### Code anatomy, fitting

| Part | Role |
|------|------|
| <span style="color:#2563eb">lm(</span><span style="color:#dc2626">fro_yos_sold ~ high_temp_f</span><span style="color:#2563eb">, data = </span><span style="color:#059669">froyo</span><span style="color:#2563eb">)</span> | Fit the least-squares line, response ~ predictor |
| <span style="color:#059669">fit</span> | A name you choose to store the fitted model |
| <span style="color:#2563eb">coef()</span> | Pull out the intercept b0 and slope b1 |
| <span style="color:#2563eb">summary()</span> | Full table: estimates, standard errors, t, p, and R-squared |

In [ ]:
fit <- lm(fro_yos_sold ~ high_temp_f, data = froyo)

coef(fit)

summary(fit)

Read the output. In <span style="color:#7c3aed">coef()</span> the (Intercept) is b0 and the high_temp_f value is the slope b1 (about 2.3). The prediction equation is

$$\widehat{\text{fro\_yos\_sold}} = b_0 + b_1 \times \text{high\_temp\_f}.$$

Slope (about 2.3 fro yos per degree): each additional degree of high temperature is associated with about 2.3 more frozen yogurts sold that day. Intercept (about -59): the predicted sales at 0 degrees, which is far outside the data, so it only anchors the height of the line and is not a meaningful prediction. In <span style="color:#7c3aed">summary()</span> the Multiple R-squared (about 0.94) and the slope's small p-value come next; we read those in Step 4.

## Step 4, Assess the model

Before trusting the model, check the conditions and measure how well it predicts. Conditions: a straight-line form (seen in Step 2), independent days, and residuals that scatter evenly around 0 with roughly constant spread. Then read R-squared and a typical prediction error.

### Code anatomy, assessing

| Part | Role |
|------|------|
| <span style="color:#2563eb">resid()</span>, <span style="color:#2563eb">fitted()</span> | The residuals (y minus yhat) and the fitted values yhat |
| <span style="color:#2563eb">plot()</span> residuals vs fitted | Look for an even band around 0 (no funnel, no curve) |
| <span style="color:#2563eb">qqnorm()</span>, <span style="color:#2563eb">qqline()</span> | Check whether residuals look roughly normal |
| <span style="color:#7c3aed">r.squared</span> | Fraction of variation in sales explained by temperature |

In [ ]:
# Residuals vs fitted: we want an even band around 0
plot(fitted(fit), resid(fit),
     xlab = "Fitted (predicted) sales", ylab = "Residual",
     main = "Residuals vs fitted")
abline(h = 0, col = "darkred", lwd = 2)

# Q-Q plot: we want points near the line
qqnorm(resid(fit)); qqline(resid(fit), col = "darkred", lwd = 2)

# R-squared: share of variation explained
summary(fit)$r.squared

# Typical prediction error in fro yos: RMSE
rmse <- sqrt(mean(resid(fit)^2))
rmse

Read the output. The residuals vs fitted plot shows points scattered fairly evenly above and below 0, with no funnel and no bend, so the constant-spread and linear-form conditions look fine. The Q-Q points lie close to the line, so the normal condition is reasonable. R-squared is about 0.94, so temperature explains about 94 percent of the day-to-day variation in sales. RMSE is about 4, so a typical prediction is off by roughly 4 frozen yogurts. With only 14 days, these numbers are measured on the same data used to fit the model, so treat them as in-sample; more days, or a held-out test set, would give a more honest error estimate.

## Step 5, Use the model to predict tomorrow

Tomorrow's forecast high is 57 degrees. We want a single best guess and a 95 percent interval.

Important: build a new data frame for the prediction. You cannot just hand `predict()` the number 57. The model was fit with a predictor column named high_temp_f, so `predict()` needs the new data in the same shape: a data frame with a column named exactly high_temp_f. That is why we write `data.frame(high_temp_f = 57)` instead of just 57. The column name is how `predict()` knows that 57 is a temperature. A bare number carries no column name, and a mismatched name like `temp = 57` will error or be ignored.

### Code anatomy, prediction

| Part | Role |
|------|------|
| <span style="color:#2563eb">data.frame(</span><span style="color:#dc2626">high_temp_f = 57</span><span style="color:#2563eb">)</span> | The new data; the column name must match the model's predictor |
| <span style="color:#2563eb">predict(</span><span style="color:#059669">fit</span><span style="color:#2563eb">, newdata = ...)</span> | Single best guess (the point prediction yhat) |
| <span style="color:#2563eb">interval = </span><span style="color:#dc2626">"prediction"</span> | 95 percent interval for one new day (what we want) |
| <span style="color:#2563eb">interval = </span><span style="color:#dc2626">"confidence"</span> | 95 percent interval for the mean of all such days (narrower) |

In [ ]:
# Tomorrow's predictor value, as a data frame with the matching column name
tomorrow <- data.frame(high_temp_f = 57)

# Single best guess (point prediction)
predict(fit, newdata = tomorrow)

# 95% prediction interval for ONE day at 57 degrees (this is what we want)
predict(fit, newdata = tomorrow, interval = "prediction", level = 0.95)

# For contrast: 95% confidence interval for the MEAN sales on all 57-degree days
predict(fit, newdata = tomorrow, interval = "confidence", level = 0.95)

Read the output. The point prediction (fit) is about 71, our single best guess for tomorrow's sales at 57 degrees. The prediction interval (interval = "prediction") gives lwr and upr for one individual day; this is the interval to report when you predict a single day, and it is the wider of the two because it includes the day-to-day variation around the line. The confidence interval (interval = "confidence") is narrower because it only covers the average sales across all 57-degree days, not one specific day.

Conclusion: at a high of 57 degrees we predict about 71 frozen yogurts tomorrow, and we are 95 percent confident the count for that single day falls in the prediction interval shown. Because 57 is inside the observed range (49 to 71), this is interpolation and the model is on solid ground; predicting a 95 degree day would be extrapolation and far less trustworthy.

## Recap of the workflow

1. Get the data into R (read_csv from a shared sheet, or upload a CSV).
2. Check it loaded correctly (glimpse, head, names, nrow).
3. Explore (summaries, the predictor's range, cor, a labeled scatterplot).
4. Build the model (lm(response ~ predictor)), interpret slope and intercept with units.
5. Assess (residual plots, R-squared, RMSE, and the conditions).
6. Use (build a data.frame for the new x, then predict with a prediction interval for one case).

You now have every step you need to run this on a dataset of your own.